## 1. Setup and Installation

In [ ]:
print('cell 1')
# =============================================================================
# CELL 1: SETUP AND INSTALLATION
# =============================================================================

# Install required packages on Kaggle
!pip install pytorch-crf --quiet

import os
import sys
import time
import json
import random
import re
import unicodedata
from datetime import datetime
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Optional, Union, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from tqdm.auto import tqdm

# Check device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Constants and Configuration

In [ ]:
print('cell 2')
# =============================================================================
# CELL 2: CONFIGURATION (P100 Optimized)
# =============================================================================

class Config:
    """Training configuration optimized for P100 GPU (16GB HBM2)."""
    # Paths (adjust for Kaggle)
    TRAIN_FILE = '/kaggle/input/arabic-diacritization/train.txt'
    VAL_FILE = '/kaggle/input/arabic-diacritization/val.txt'
    OUTPUT_DIR = '/kaggle/working/'
    
    # Model Architecture
    ARABERT_MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
    ARABERT_HIDDEN_SIZE = 768
    ARABERT_NUM_LAYERS = 4  # Last N layers to use
    ARABERT_AGGREGATION = 'mean'
    
    LSTM_HIDDEN_SIZE = 512
    LSTM_NUM_LAYERS = 2      # 2 layers for efficient training
    LSTM_DROPOUT = 0.3
    LSTM_BIDIRECTIONAL = True
    
    # Sequence Settings
    MAX_SEQ_LENGTH = 512  # Full sequence length
    MIN_SEQ_LENGTH = 5
    
    # Training Hyperparameters
    BATCH_SIZE = 128             # Larger batch size for GPU utilization
    GRADIENT_ACCUMULATION = 1    # No accumulation needed
    NUM_EPOCHS = 2               # Reduced epochs for bottleneck testing
    
    # Differential Learning Rates
    ARABERT_LR = 2e-5      # Lower LR for pretrained weights
    LSTM_CRF_LR = 1e-3     # Higher LR for new layers
    
    WEIGHT_DECAY = 1e-5
    WARMUP_RATIO = 0.1     # 10% warmup
    GRADIENT_CLIP = 1.0
    
    # Early Stopping
    EARLY_STOPPING_PATIENCE = 7
    
    # Mixed Precision
    USE_FP16 = True
    
    # Reproducibility
    SEED = 42
    
    # Logging
    LOG_INTERVAL = 50  # Print timing every N batches
    
    # DataLoader
    NUM_WORKERS = 4  # Increased for faster data loading
    PREFETCH_FACTOR = 2  # Prefetch batches for better GPU utilization
    
    # Bucketed Batching
    NUM_BUCKETS = 8  # Number of length buckets
    BUCKET_BOUNDARIES = [64, 128, 192, 256, 320, 384, 448, 512]  # Fixed bucket boundaries

config = Config()

print("Configuration (Optimized):")
print(f"  MAX_SEQ_LENGTH: {config.MAX_SEQ_LENGTH}")
print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
print(f"  NUM_BUCKETS: {config.NUM_BUCKETS}")
print(f"  BUCKET_BOUNDARIES: {config.BUCKET_BOUNDARIES}")

In [ ]:
print('cell 3')
# =============================================================================
# CELL 3: ARABIC DIACRITICS DEFINITIONS
# =============================================================================

# Individual diacritics (Unicode)
FATHA = '\u064E'       # َ  - open 'a' sound
DAMMA = '\u064F'       # ُ  - 'u' sound
KASRA = '\u0650'       # ِ  - 'i' sound
SUKUN = '\u0652'       # ْ  - no vowel (silence)
SHADDA = '\u0651'      # ّ  - consonant doubling

# Tanwin (nunation)
FATHATAN = '\u064B'    # ً  - 'an' sound
DAMMATAN = '\u064C'    # ٌ  - 'un' sound
KASRATAN = '\u064D'    # ٍ  - 'in' sound

# Additional diacritics
SUPERSCRIPT_ALEF = '\u0670'  # ٰ
MADDAH = '\u0653'            # ٓ

# All diacritics for stripping
ARABIC_DIACRITICS = [FATHATAN, DAMMATAN, KASRATAN, FATHA, DAMMA, KASRA, SHADDA, SUKUN]
ARABIC_DIACRITICS_EXTENDED = ARABIC_DIACRITICS + [SUPERSCRIPT_ALEF, MADDAH]

# 16 Diacritic Labels for Classification
DIACRITIC_LABELS = [
    '',                      # 0: NO_DIACRITIC
    FATHA,                   # 1: Fatha
    DAMMA,                   # 2: Damma
    KASRA,                   # 3: Kasra
    SUKUN,                   # 4: Sukun
    FATHATAN,                # 5: Fathatan
    DAMMATAN,                # 6: Dammatan
    KASRATAN,                # 7: Kasratan
    SHADDA,                  # 8: Shadda alone
    SHADDA + FATHA,          # 9: Shadda + Fatha
    SHADDA + DAMMA,          # 10: Shadda + Damma
    SHADDA + KASRA,          # 11: Shadda + Kasra
    SHADDA + SUKUN,          # 12: Shadda + Sukun
    SHADDA + FATHATAN,       # 13: Shadda + Fathatan
    SHADDA + DAMMATAN,       # 14: Shadda + Dammatan
    SHADDA + KASRATAN,       # 15: Shadda + Kasratan
]

NUM_DIACRITIC_CLASSES = len(DIACRITIC_LABELS)  # 16

# Create mappings
DIACRITIC_TO_ID = {diac: idx for idx, diac in enumerate(DIACRITIC_LABELS)}
ID_TO_DIACRITIC = {idx: diac for idx, diac in enumerate(DIACRITIC_LABELS)}

# Label names for logging
LABEL_NAMES = [
    'NO_DIACRITIC', 'FATHA', 'DAMMA', 'KASRA', 'SUKUN',
    'FATHATAN', 'DAMMATAN', 'KASRATAN', 'SHADDA',
    'SHADDA_FATHA', 'SHADDA_DAMMA', 'SHADDA_KASRA', 'SHADDA_SUKUN',
    'SHADDA_FATHATAN', 'SHADDA_DAMMATAN', 'SHADDA_KASRATAN',
]

# Special tokens
PAD_ID = 0

print(f"Number of diacritic classes: {NUM_DIACRITIC_CLASSES}")

## 3. Utility Functions

In [ ]:
print('cell 4')
# =============================================================================
# CELL 4: REPRODUCIBILITY & UTILITIES
# =============================================================================

def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(config.SEED)
print(f"Random seed set to {config.SEED}")


class AverageMeter:
    """Computes and stores the average and current value."""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0


class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience: int = 5, mode: str = 'min', min_delta: float = 0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score: float, epoch: int) -> bool:
        if self.mode == 'min':
            score = -score
        
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return True
        
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False

## 4. Preprocessing Functions

In [ ]:
print('cell 5')
# =============================================================================
# CELL 5: TEXT PREPROCESSING
# =============================================================================

def is_arabic_diacritic(char: str) -> bool:
    """Check if a character is an Arabic diacritic."""
    return char in ARABIC_DIACRITICS_EXTENDED


def is_arabic_letter(char: str) -> bool:
    """Check if a character is an Arabic letter."""
    code = ord(char)
    return (0x0621 <= code <= 0x064A) or (0x0671 <= code <= 0x06D3)


def strip_diacritics(text: str) -> str:
    """Remove all diacritics from Arabic text."""
    for diac in ARABIC_DIACRITICS_EXTENDED:
        text = text.replace(diac, '')
    return text


def clean_text(text: str) -> str:
    """Clean Arabic text by removing unwanted characters."""
    if not text:
        return text
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove control characters
    text = ''.join(char for char in text if unicodedata.category(char) != 'Cc' or char in '\n\t ')
    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    # Remove Tatweel (elongation)
    text = text.replace('\u0640', '')
    
    return text.strip()


def align_diacritics(diacritized_text: str) -> Tuple[List[str], List[int]]:
    """
    Align diacritics with base characters.
    
    Returns:
        chars: List of base characters (no diacritics)
        labels: List of diacritic label IDs
    """
    chars = []
    labels = []
    
    i = 0
    n = len(diacritized_text)
    
    while i < n:
        char = diacritized_text[i]
        
        # Skip if current char is a diacritic
        if is_arabic_diacritic(char):
            i += 1
            continue
        
        # This is a base character
        chars.append(char)
        
        # Collect all diacritics following this character
        diacritic_str = ''
        j = i + 1
        while j < n and is_arabic_diacritic(diacritized_text[j]):
            diacritic_str += diacritized_text[j]
            j += 1
        
        # Normalize: Shadda should come first
        if SHADDA in diacritic_str and len(diacritic_str) > 1:
            other_diac = diacritic_str.replace(SHADDA, '')
            diacritic_str = SHADDA + other_diac
        
        # Convert to label ID
        if diacritic_str in DIACRITIC_TO_ID:
            label_id = DIACRITIC_TO_ID[diacritic_str]
        else:
            label_id = DIACRITIC_TO_ID['']  # Unknown -> no diacritic
        
        labels.append(label_id)
        i = j if j > i + 1 else i + 1
    
    return chars, labels


def reconstruct_diacritized_text(chars: List[str], labels: List[int]) -> str:
    """Reconstruct diacritized text from characters and labels."""
    result = []
    for char, label_id in zip(chars, labels):
        result.append(char)
        diacritic = ID_TO_DIACRITIC.get(label_id, '')
        if diacritic:
            result.append(diacritic)
    return ''.join(result)

## 5. Data Loading

In [ ]:
print('cell 6')
# =============================================================================
# CELL 6: DATA LOADING AND PROCESSING
# =============================================================================

def load_sentences(filepath: str) -> List[str]:
    """Load sentences from a text file (one sentence per line)."""
    sentences = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                sentences.append(line)
    return sentences


def process_dataset(filepath: str, 
                    max_length: int = None,
                    min_length: int = None,
                    verbose: bool = True) -> List[Dict]:
    """
    Process dataset file into training format.
    
    Returns:
        List of dicts with 'chars', 'labels', 'text_undiacritized'
    """
    if max_length is None:
        max_length = config.MAX_SEQ_LENGTH
    if min_length is None:
        min_length = config.MIN_SEQ_LENGTH
    
    sentences = load_sentences(filepath)
    if verbose:
        print(f"Loaded {len(sentences)} sentences from {filepath}")
    
    processed_data = []
    skipped_short = 0
    truncated = 0
    
    for sentence in tqdm(sentences, desc="Processing", disable=not verbose):
        # Clean text
        sentence = clean_text(sentence)
        if not sentence:
            continue
        
        # Align diacritics
        chars, labels = align_diacritics(sentence)
        
        # Skip too short
        if len(chars) < min_length:
            skipped_short += 1
            continue
        
        # Truncate if needed
        if len(chars) > max_length:
            chars = chars[:max_length]
            labels = labels[:max_length]
            truncated += 1
        
        entry = {
            'chars': chars,
            'labels': labels,
            'text_undiacritized': ''.join(chars),
        }
        processed_data.append(entry)
    
    if verbose:
        print(f"  Processed: {len(processed_data)} sentences")
        print(f"  Skipped (too short): {skipped_short}")
        print(f"  Truncated: {truncated}")
    
    return processed_data

In [ ]:
print('cell 7')
# =============================================================================
# CELL 7: PYTORCH DATASET
# =============================================================================

class DiacritizationDataset(Dataset):
    """PyTorch Dataset for Arabic diacritization."""
    
    def __init__(self, data: List[Dict], max_length: int = None):
        self.data = data
        self.max_length = max_length or config.MAX_SEQ_LENGTH
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, idx: int) -> Dict:
        entry = self.data[idx]
        return {
            'chars': entry['chars'][:self.max_length],
            'labels': entry['labels'][:self.max_length],
            'text': entry['text_undiacritized'][:self.max_length],
            'length': min(len(entry['chars']), self.max_length)
        }

In [ ]:
print('cell 8')
# =============================================================================
# CELL 8: LOAD DATA
# =============================================================================

# For local testing, update paths
import os
if not os.path.exists(config.TRAIN_FILE):
    # Local paths for testing
    config.TRAIN_FILE = 'dataset/train.txt'
    config.VAL_FILE = 'dataset/val.txt'
    config.OUTPUT_DIR = './'

print("Loading training data...")
train_data = process_dataset(config.TRAIN_FILE, verbose=True)

print("\nLoading validation data...")
val_data = process_dataset(config.VAL_FILE, verbose=True)

# Create datasets
train_dataset = DiacritizationDataset(train_data)
val_dataset = DiacritizationDataset(val_data)

print(f"\nDataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}")

# Show sample
sample = train_dataset[0]
print(f"\nSample:")
print(f"  Text: {sample['text'][:50]}...")
print(f"  Length: {sample['length']} characters")
print(f"  Labels: {sample['labels'][:10]}...")

---
## ✅ Step 1 Complete

**What we have so far:**
- Setup and imports
- Configuration class with all hyperparameters
- Arabic diacritics definitions (16 classes)
- Utility functions (seed, early stopping, meters)
- Preprocessing functions (cleaning, diacritic alignment)
- Data loading and PyTorch Dataset

---
# STEP 2: Feature Extraction & Model Architecture

## 6. AraBERT Feature Extractor

In [ ]:
print('cell 9')
# =============================================================================
# CELL 9: ARABERT FEATURE EXTRACTOR
# =============================================================================

class AraBERTFeatureExtractor(nn.Module):
    """
    Extract contextual embeddings from AraBERT for each character.
    
    - Uses aubmindlab/bert-base-arabertv02
    - Extracts last N hidden layers with aggregation
    - Maps subword tokens back to character-level embeddings
    """
    
    def __init__(self, 
                 model_name: str = None,
                 num_layers: int = 4,
                 aggregation: str = 'mean',  # 'mean', 'concat', 'last'
                 dropout: float = 0.3,
                 freeze_epochs: int = 0):  # Freeze BERT for first N epochs
        super().__init__()
        
        if model_name is None:
            model_name = config.ARABERT_MODEL_NAME
        
        self.model_name = model_name
        self.num_layers = num_layers
        self.aggregation = aggregation
        self.freeze_epochs = freeze_epochs
        self.current_epoch = 0
        
        print(f"Loading AraBERT: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        
        # Dropout after BERT output
        self.dropout = nn.Dropout(dropout)
        
        # Output dimension
        if aggregation == 'concat':
            self.output_dim = config.ARABERT_HIDDEN_SIZE * num_layers
        else:
            self.output_dim = config.ARABERT_HIDDEN_SIZE
        
        print(f"AraBERT loaded. Output dim: {self.output_dim}")
    
    def set_epoch(self, epoch: int):
        """Update current epoch for freeze/unfreeze logic."""
        self.current_epoch = epoch
        if epoch < self.freeze_epochs:
            self._freeze_bert()
        else:
            self._unfreeze_bert()
    
    def _freeze_bert(self):
        """Freeze BERT parameters."""
        for param in self.bert.parameters():
            param.requires_grad = False
    
    def _unfreeze_bert(self):
        """Unfreeze BERT parameters."""
        for param in self.bert.parameters():
            param.requires_grad = True
    
    def forward(self, texts: List[str], max_char_len: int = None) -> torch.Tensor:
        """
        Extract AraBERT features for a batch of texts.
        
        Args:
            texts: List of undiacritized text strings
            max_char_len: Maximum character length (for padding)
            
        Returns:
            Tensor of shape (batch_size, max_char_len, output_dim)
        """
        batch_size = len(texts)
        device = next(self.bert.parameters()).device
        
        if max_char_len is None:
            max_char_len = max(len(t) for t in texts)
        
        # Tokenize all texts
        encoding = self.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512,
            return_offsets_mapping=True
        )
        
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        offset_mappings = encoding['offset_mapping']  # (batch, num_tokens, 2)
        
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states  # Tuple of (batch, seq_len, hidden)
        
        # Get last N layers
        last_n_layers = hidden_states[-self.num_layers:]
        
        # Aggregate layers
        if self.aggregation == 'concat':
            token_embeddings = torch.cat(last_n_layers, dim=-1)
        elif self.aggregation == 'mean':
            stacked = torch.stack(last_n_layers, dim=0)
            token_embeddings = stacked.mean(dim=0)
        else:  # 'last'
            token_embeddings = last_n_layers[-1]
        
        # Apply dropout
        token_embeddings = self.dropout(token_embeddings)
        
        # Map tokens to characters for each sample
        char_embeddings = torch.zeros(batch_size, max_char_len, self.output_dim, device=device)
        
        for i in range(batch_size):
            text = texts[i]
            num_chars = min(len(text), max_char_len)
            offsets = offset_mappings[i].tolist()
            
            char_counts = torch.zeros(num_chars, device=device)
            
            for token_idx, (start, end) in enumerate(offsets):
                if start == end:  # Special token
                    continue
                
                for char_idx in range(start, min(end, num_chars)):
                    char_embeddings[i, char_idx] += token_embeddings[i, token_idx]
                    char_counts[char_idx] += 1
            
            # Average for characters covered by multiple tokens
            char_counts = char_counts.clamp(min=1)
            char_embeddings[i, :num_chars] /= char_counts.unsqueeze(-1)
        
        return char_embeddings
    
    def get_output_dim(self) -> int:
        return self.output_dim

## 7. Character Features Extractor

In [ ]:
print('cell 10')
# =============================================================================
# CELL 10: CHARACTER-LEVEL FEATURES
# =============================================================================

# Arabic Character Categories
VOWEL_CARRIERS = set('اوي')
SUN_LETTERS = set('تثدذرزسشصضطظلن')
MOON_LETTERS = set('ابجحخعغفقكمهوي')
EMPHATIC_CONSONANTS = set('صضطظ')
HAMZA_FORMS = set('ءأإؤئ')
ALEF_FORMS = set('اأإآٱى')
TEH_MARBUTA = 'ة'


class CharacterFeatureExtractor(nn.Module):
    """
    Extract character-type features and learned embeddings.
    
    Features per character:
    - Vowel carrier (3 dim)
    - Sun/Moon letter (2 dim)
    - Emphatic consonant (1 dim)
    - Hamza form (1 dim)
    - Teh Marbuta (1 dim)
    - Alef form (1 dim)
    - Is space/punct (1 dim)
    - Position in word (3 dim)
    - Relative position (1 dim)
    - Word length category (4 dim)
    - Learned character embeddings (64 dim)
    
    Total: ~82 dimensions
    """
    
    CHAR_TYPE_DIM = 18
    CHAR_EMBED_DIM = 64
    
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Build character vocabulary
        self.char_to_idx = self._build_vocab()
        self.vocab_size = len(self.char_to_idx)
        
        # Learnable character embeddings
        self.char_embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        
        self.output_dim = self.CHAR_TYPE_DIM + embed_dim
    
    def _build_vocab(self) -> Dict[str, int]:
        """Build character vocabulary."""
        vocab = {'<PAD>': 0, '<UNK>': 1, ' ': 2}
        
        # Arabic letters
        for code in range(0x0621, 0x064B):
            vocab[chr(code)] = len(vocab)
        
        # Additional characters
        for char in 'ٱىة،.؛؟!,;:()[]{}"\'-':
            if char not in vocab:
                vocab[char] = len(vocab)
        
        return vocab
    
    def _get_word_positions(self, chars: List[str]) -> List[Tuple[int, int]]:
        """Get word boundaries for each character."""
        positions = []
        word_start = 0
        separators = set(' ،.؛؟!,;:')
        
        for i, char in enumerate(chars):
            if char in separators:
                positions.append((i, i + 1))
                word_start = i + 1
            else:
                word_end = i + 1
                while word_end < len(chars) and chars[word_end] not in separators:
                    word_end += 1
                positions.append((word_start, word_end))
        
        return positions
    
    def _extract_char_type_features(self, chars: List[str]) -> torch.Tensor:
        """Extract character-type features."""
        seq_len = len(chars)
        features = torch.zeros(seq_len, self.CHAR_TYPE_DIM)
        
        word_positions = self._get_word_positions(chars)
        
        for i, char in enumerate(chars):
            feat_idx = 0
            
            # Vowel carrier (3 dim)
            if char == 'ا':
                features[i, feat_idx] = 1.0
            elif char == 'و':
                features[i, feat_idx + 1] = 1.0
            elif char == 'ي':
                features[i, feat_idx + 2] = 1.0
            feat_idx += 3
            
            # Sun letter (1 dim)
            features[i, feat_idx] = 1.0 if char in SUN_LETTERS else 0.0
            feat_idx += 1
            
            # Moon letter (1 dim)
            features[i, feat_idx] = 1.0 if char in MOON_LETTERS else 0.0
            feat_idx += 1
            
            # Emphatic consonant (1 dim)
            features[i, feat_idx] = 1.0 if char in EMPHATIC_CONSONANTS else 0.0
            feat_idx += 1
            
            # Hamza form (1 dim)
            features[i, feat_idx] = 1.0 if char in HAMZA_FORMS else 0.0
            feat_idx += 1
            
            # Teh Marbuta (1 dim)
            features[i, feat_idx] = 1.0 if char == TEH_MARBUTA else 0.0
            feat_idx += 1
            
            # Alef form (1 dim)
            features[i, feat_idx] = 1.0 if char in ALEF_FORMS else 0.0
            feat_idx += 1
            
            # Is space/punctuation (1 dim)
            features[i, feat_idx] = 1.0 if not is_arabic_letter(char) else 0.0
            feat_idx += 1
            
            # Position in word (3 dim: first, middle, last)
            word_start, word_end = word_positions[i]
            word_len = word_end - word_start
            pos_in_word = i - word_start
            
            if word_len == 1:
                features[i, feat_idx] = 1.0      # first
                features[i, feat_idx + 2] = 1.0  # last
            elif pos_in_word == 0:
                features[i, feat_idx] = 1.0      # first
            elif pos_in_word == word_len - 1:
                features[i, feat_idx + 2] = 1.0  # last
            else:
                features[i, feat_idx + 1] = 1.0  # middle
            feat_idx += 3
            
            # Relative position (1 dim)
            features[i, feat_idx] = pos_in_word / max(word_len - 1, 1)
            feat_idx += 1
            
            # Word length category (4 dim)
            if word_len <= 2:
                features[i, feat_idx] = 1.0
            elif word_len <= 4:
                features[i, feat_idx + 1] = 1.0
            elif word_len <= 6:
                features[i, feat_idx + 2] = 1.0
            else:
                features[i, feat_idx + 3] = 1.0
        
        return features
    
    def forward(self, chars_batch: List[List[str]], max_len: int = None) -> torch.Tensor:
        """
        Extract features for a batch of character sequences.
        
        Args:
            chars_batch: List of character lists
            max_len: Maximum sequence length
            
        Returns:
            Tensor of shape (batch_size, max_len, output_dim)
        """
        batch_size = len(chars_batch)
        device = next(self.parameters()).device
        
        if max_len is None:
            max_len = max(len(chars) for chars in chars_batch)
        
        # Character type features
        type_features = torch.zeros(batch_size, max_len, self.CHAR_TYPE_DIM, device=device)
        
        # Character indices for embeddings
        char_indices = torch.zeros(batch_size, max_len, dtype=torch.long, device=device)
        
        for i, chars in enumerate(chars_batch):
            seq_len = min(len(chars), max_len)
            
            # Type features
            type_feat = self._extract_char_type_features(chars[:seq_len])
            type_features[i, :seq_len] = type_feat.to(device)
            
            # Character indices
            indices = [self.char_to_idx.get(c, 1) for c in chars[:seq_len]]  # 1 = <UNK>
            char_indices[i, :seq_len] = torch.tensor(indices, dtype=torch.long)
        
        # Get character embeddings
        char_embeds = self.char_embedding(char_indices)
        
        # Concatenate
        features = torch.cat([type_features, char_embeds], dim=-1)
        
        return features
    
    def get_output_dim(self) -> int:
        return self.output_dim

## 8. BiLSTM-CRF Model

In [ ]:
print('cell 11')
# =============================================================================
# CELL 11: BiLSTM-CRF MODEL
# =============================================================================

class BiLSTM_CRF(nn.Module):
    """
    Bidirectional LSTM with CRF layer for sequence labeling.
    
    Architecture:
        Input Features → Input Projection → BiLSTM → LayerNorm → Linear → CRF
    """
    
    def __init__(self,
                 input_dim: int,
                 hidden_dim: int = 512,
                 num_layers: int = 2,
                 num_classes: int = 16,
                 dropout: float = 0.3,
                 bidirectional: bool = True):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_classes = num_classes
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.input_dropout = nn.Dropout(dropout)
        
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Output layers
        lstm_output_dim = hidden_dim * self.num_directions
        self.layer_norm = nn.LayerNorm(lstm_output_dim)
        self.output_dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(lstm_output_dim, num_classes)
        
        # CRF layer
        self.crf = CRF(num_classes, batch_first=True)
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights."""
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def _get_lstm_features(self, features: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """Pass features through BiLSTM to get emissions."""
        batch_size, seq_len, _ = features.shape
        
        # Project input
        features = self.input_projection(features)
        features = self.input_dropout(features)
        
        # LSTM with packing for efficiency
        if mask is not None:
            lengths = mask.sum(dim=1).cpu()
            packed = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=False)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=seq_len)
        else:
            lstm_out, _ = self.lstm(features)
        
        # Output processing
        lstm_out = self.layer_norm(lstm_out)
        lstm_out = self.output_dropout(lstm_out)
        emissions = self.hidden2tag(lstm_out)
        
        return emissions
    
    def forward(self, features: torch.Tensor, labels: torch.Tensor = None, 
                mask: torch.Tensor = None) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            features: (batch, seq_len, input_dim)
            labels: (batch, seq_len) - ground truth labels
            mask: (batch, seq_len) - boolean mask
            
        Returns:
            If labels provided: CRF negative log-likelihood loss
            If no labels: emissions (batch, seq_len, num_classes)
        """
        emissions = self._get_lstm_features(features, mask)
        
        if labels is not None:
            # Training: compute CRF loss
            if mask is not None:
                mask = mask.bool()
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            return emissions
    
    def decode(self, features: torch.Tensor, mask: torch.Tensor = None) -> List[List[int]]:
        """Decode best label sequence using Viterbi algorithm."""
        emissions = self._get_lstm_features(features, mask)
        
        if mask is not None:
            mask = mask.bool()
        
        return self.crf.decode(emissions, mask=mask)

## 9. Complete Diacritization Model (AraBERT + BiLSTM-CRF)

In [ ]:
print('cell 12')
# =============================================================================
# CELL 12: COMPLETE DIACRITIZATION MODEL
# =============================================================================

class ArabicDiacritizationModel(nn.Module):
    """
    Complete Arabic Diacritization Model.
    
    Architecture:
        AraBERT (768 dim) + Character Features (82 dim)
        → BiLSTM (2 layers, 512 hidden, bidirectional → 1024)
        → Linear (1024 → 16)
        → CRF
    """
    
    def __init__(self,
                 arabert_model: str = None,
                 hidden_dim: int = 512,
                 num_layers: int = 2,
                 num_classes: int = 16,
                 dropout: float = 0.3,
                 use_char_features: bool = True):
        super().__init__()
        
        if arabert_model is None:
            arabert_model = config.ARABERT_MODEL_NAME
        
        self.use_char_features = use_char_features
        
        # Feature extractors
        self.arabert = AraBERTFeatureExtractor(
            model_name=arabert_model,
            num_layers=config.ARABERT_NUM_LAYERS,
            aggregation='mean',
            dropout=dropout
        )
        
        if use_char_features:
            self.char_features = CharacterFeatureExtractor(embed_dim=64)
            input_dim = self.arabert.get_output_dim() + self.char_features.get_output_dim()
        else:
            self.char_features = None
            input_dim = self.arabert.get_output_dim()
        
        print(f"Total input dimension: {input_dim}")
        
        # BiLSTM-CRF
        self.bilstm_crf = BiLSTM_CRF(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=num_classes,
            dropout=dropout,
            bidirectional=True
        )
        
        self.num_classes = num_classes
    
    def set_epoch(self, epoch: int):
        """Update epoch for AraBERT freeze/unfreeze."""
        self.arabert.set_epoch(epoch)
    
    def forward(self, 
                texts: List[str], 
                chars_batch: List[List[str]],
                labels: torch.Tensor = None,
                mask: torch.Tensor = None) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            texts: List of undiacritized text strings
            chars_batch: List of character lists
            labels: (batch, seq_len) ground truth
            mask: (batch, seq_len) boolean mask
            
        Returns:
            Loss if labels provided, else emissions
        """
        batch_size = len(texts)
        max_len = max(len(chars) for chars in chars_batch)
        
        # AraBERT features
        arabert_features = self.arabert(texts, max_char_len=max_len)
        
        # Character features
        if self.use_char_features:
            char_features = self.char_features(chars_batch, max_len=max_len)
            features = torch.cat([arabert_features, char_features], dim=-1)
        else:
            features = arabert_features
        
        # BiLSTM-CRF
        return self.bilstm_crf(features, labels, mask)
    
    def decode(self, texts: List[str], chars_batch: List[List[str]], 
               mask: torch.Tensor = None) -> List[List[int]]:
        """Decode predictions using Viterbi."""
        batch_size = len(texts)
        max_len = max(len(chars) for chars in chars_batch)
        
        # Get features
        arabert_features = self.arabert(texts, max_char_len=max_len)
        
        if self.use_char_features:
            char_features = self.char_features(chars_batch, max_len=max_len)
            features = torch.cat([arabert_features, char_features], dim=-1)
        else:
            features = arabert_features
        
        return self.bilstm_crf.decode(features, mask)
    
    def get_parameter_groups(self, arabert_lr: float, other_lr: float) -> List[Dict]:
        """
        Get parameter groups with differential learning rates.
        
        Args:
            arabert_lr: Learning rate for AraBERT parameters
            other_lr: Learning rate for LSTM/CRF/other parameters
            
        Returns:
            List of parameter group dicts for optimizer
        """
        arabert_params = []
        other_params = []
        
        for name, param in self.named_parameters():
            if 'arabert' in name or 'bert' in name:
                arabert_params.append(param)
            else:
                other_params.append(param)
        
        return [
            {'params': arabert_params, 'lr': arabert_lr},
            {'params': other_params, 'lr': other_lr}
        ]

## 10. DataLoader with Collate Function

In [ ]:
print('cell 13')
# =============================================================================
# CELL 13: DATALOADER WITH BUCKETED BATCHING
# =============================================================================

from torch.utils.data import Sampler

class BucketBatchSampler(Sampler):
    """
    Batch sampler that groups sequences of similar lengths together.
    
    This minimizes padding within batches, improving GPU throughput by:
    - Reducing wasted computation on padding tokens
    - Ensuring more uniform batch processing times
    
    Args:
        lengths: List of sequence lengths for each sample
        batch_size: Number of samples per batch
        bucket_boundaries: List of length boundaries for buckets
        shuffle: Whether to shuffle within buckets (True for training)
        drop_last: Whether to drop the last incomplete batch
    """
    
    def __init__(self, 
                 lengths: List[int], 
                 batch_size: int,
                 bucket_boundaries: List[int] = None,
                 shuffle: bool = True,
                 drop_last: bool = False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        
        # Default bucket boundaries if not provided
        if bucket_boundaries is None:
            bucket_boundaries = [64, 128, 192, 256]
        self.bucket_boundaries = sorted(bucket_boundaries)
        
        # Assign each sample to a bucket based on its length
        self.buckets = self._create_buckets()
        
        # Pre-compute batches
        self.batches = self._create_batches()
    
    def _get_bucket_id(self, length: int) -> int:
        """Get bucket ID for a given sequence length."""
        for i, boundary in enumerate(self.bucket_boundaries):
            if length <= boundary:
                return i
        return len(self.bucket_boundaries)  # Last bucket for longer sequences
    
    def _create_buckets(self) -> Dict[int, List[int]]:
        """Assign sample indices to buckets based on length."""
        buckets = defaultdict(list)
        for idx, length in enumerate(self.lengths):
            bucket_id = self._get_bucket_id(length)
            buckets[bucket_id].append(idx)
        return buckets
    
    def _create_batches(self) -> List[List[int]]:
        """Create batches from bucketed samples."""
        batches = []
        
        for bucket_id, indices in self.buckets.items():
            # Shuffle indices within bucket if needed
            if self.shuffle:
                indices = indices.copy()
                random.shuffle(indices)
            
            # Create batches from this bucket
            for i in range(0, len(indices), self.batch_size):
                batch = indices[i:i + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)
        
        # Shuffle batches across buckets if training
        if self.shuffle:
            random.shuffle(batches)
        
        return batches
    
    def __iter__(self):
        # Re-create batches each epoch for different shuffling
        if self.shuffle:
            self.batches = self._create_batches()
        
        for batch in self.batches:
            yield batch
    
    def __len__(self) -> int:
        return len(self.batches)


def collate_fn(batch: List[Dict]) -> Dict:
    """
    Collate function for DataLoader.
    Pads sequences and creates tensors.
    
    With bucketed batching, sequences in each batch have similar lengths,
    so padding overhead is minimized.
    """
    batch_size = len(batch)
    max_len = max(sample['length'] for sample in batch)
    
    # Initialize tensors
    labels = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    
    chars_batch = []
    texts_batch = []
    
    for i, sample in enumerate(batch):
        seq_len = sample['length']
        labels[i, :seq_len] = torch.tensor(sample['labels'], dtype=torch.long)
        mask[i, :seq_len] = True
        chars_batch.append(sample['chars'])
        texts_batch.append(sample['text'])
    
    return {
        'chars': chars_batch,
        'texts': texts_batch,
        'labels': labels,
        'mask': mask,
    }


def create_dataloaders(train_dataset: Dataset, 
                       val_dataset: Dataset,
                       batch_size: int = 64,
                       num_workers: int = 4,
                       bucket_boundaries: List[int] = None,
                       prefetch_factor: int = 2) -> Tuple[DataLoader, DataLoader]:
    """
    Create train and validation DataLoaders with bucketed batching.
    
    Bucketed batching groups sequences of similar lengths together,
    reducing padding overhead and improving GPU utilization.
    """
    if bucket_boundaries is None:
        bucket_boundaries = config.BUCKET_BOUNDARIES
    
    # Extract lengths from datasets
    train_lengths = [sample['length'] for sample in train_dataset]
    val_lengths = [sample['length'] for sample in val_dataset]
    
    # Create bucketed samplers
    train_sampler = BucketBatchSampler(
        lengths=train_lengths,
        batch_size=batch_size,
        bucket_boundaries=bucket_boundaries,
        shuffle=True,
        drop_last=False
    )
    
    val_sampler = BucketBatchSampler(
        lengths=val_lengths,
        batch_size=batch_size,
        bucket_boundaries=bucket_boundaries,
        shuffle=False,  # No shuffle for validation
        drop_last=False
    )
    
    # Create DataLoaders with batch_sampler (not batch_size + shuffle)
    train_loader = DataLoader(
        train_dataset,
        batch_sampler=train_sampler,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False,
        prefetch_factor=prefetch_factor if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_sampler=val_sampler,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False,
        prefetch_factor=prefetch_factor if num_workers > 0 else None
    )
    
    # Print bucket statistics
    print("\nBucketed Batching Statistics:")
    print(f"  Bucket boundaries: {bucket_boundaries}")
    
    train_bucket_counts = defaultdict(int)
    for length in train_lengths:
        for i, boundary in enumerate(bucket_boundaries):
            if length <= boundary:
                train_bucket_counts[f"0-{boundary}"] += 1
                break
        else:
            train_bucket_counts[f">{bucket_boundaries[-1]}"] += 1
    
    print(f"  Train distribution by bucket:")
    for bucket, count in sorted(train_bucket_counts.items()):
        print(f"    {bucket}: {count} samples ({count/len(train_lengths)*100:.1f}%)")
    
    return train_loader, val_loader


# Create DataLoaders
print("Creating DataLoaders with Bucketed Batching...")
train_loader, val_loader = create_dataloaders(
    train_dataset, 
    val_dataset,
    batch_size=config.BATCH_SIZE,
    num_workers=config.NUM_WORKERS,
    bucket_boundaries=config.BUCKET_BOUNDARIES,
    prefetch_factor=config.PREFETCH_FACTOR
)
print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

---
## ✅ Step 2 Complete

**What we added:**
- AraBERT Feature Extractor (768 dim, character-level mapping)
- Character Feature Extractor (type features + learned embeddings = 82 dim)
- BiLSTM-CRF Model (2 layers, 512 hidden, bidirectional)
- Complete Diacritization Model combining all components
- DataLoader with collate function

---
# STEP 3: Training Loop, Evaluation & Execution

## 11. Evaluation Metrics (DER)

In [ ]:
print('cell 14')
# =============================================================================
# CELL 14: EVALUATION METRICS - Diacritic Error Rate (DER)
# =============================================================================

def compute_der(predictions: List[int], 
                targets: List[int], 
                ignore_no_diacritic: bool = True) -> Tuple[float, int, int]:
    """
    Compute Diacritic Error Rate (DER) for a single sequence.
    
    DER = (Number of diacritic errors) / (Total diacritic positions)
    
    Args:
        predictions: Predicted label sequence
        targets: Ground truth label sequence
        ignore_no_diacritic: Only count positions where target has a diacritic
        
    Returns:
        (DER, num_errors, total_positions)
    """
    if len(predictions) != len(targets):
        min_len = min(len(predictions), len(targets))
        predictions = predictions[:min_len]
        targets = targets[:min_len]
    
    errors = 0
    total = 0
    no_diacritic_id = DIACRITIC_TO_ID['']
    
    for pred, target in zip(predictions, targets):
        if ignore_no_diacritic:
            if target != no_diacritic_id:
                total += 1
                if pred != target:
                    errors += 1
        else:
            total += 1
            if pred != target:
                errors += 1
    
    der = errors / total if total > 0 else 0.0
    return der, errors, total


class Evaluator:
    """Comprehensive evaluator for diacritization models."""
    
    def __init__(self, ignore_no_diacritic: bool = True):
        self.ignore_no_diacritic = ignore_no_diacritic
        self.reset()
    
    def reset(self):
        self.all_predictions = []
        self.all_targets = []
    
    def add_batch(self, predictions: List[List[int]], targets: List[List[int]], 
                  masks: List[List[bool]] = None):
        """Add a batch of predictions and targets."""
        for i, (preds, tgts) in enumerate(zip(predictions, targets)):
            if masks is not None:
                mask = masks[i]
                preds = [p for p, m in zip(preds, mask) if m]
                tgts = [t for t, m in zip(tgts, mask) if m]
            self.all_predictions.append(preds)
            self.all_targets.append(tgts)
    
    def compute_metrics(self) -> Dict:
        """Compute all metrics."""
        total_errors = 0
        total_diacritics = 0
        total_correct = 0
        total_chars = 0
        
        for preds, tgts in zip(self.all_predictions, self.all_targets):
            _, errors, diacritics = compute_der(preds, tgts, self.ignore_no_diacritic)
            total_errors += errors
            total_diacritics += diacritics
            
            # Accuracy (all positions)
            for p, t in zip(preds, tgts):
                total_chars += 1
                if p == t:
                    total_correct += 1
        
        der = total_errors / total_diacritics if total_diacritics > 0 else 0.0
        accuracy = total_correct / total_chars if total_chars > 0 else 0.0
        
        return {
            'der': der,
            'accuracy': accuracy,
            'total_errors': total_errors,
            'total_diacritics': total_diacritics,
            'num_sequences': len(self.all_predictions)
        }


def print_metrics(metrics: Dict, title: str = "Evaluation"):
    """Print formatted metrics."""
    print(f"\n{'='*50}")
    print(f"{title}")
    print(f"{'='*50}")
    print(f"  DER:      {metrics['der']*100:.2f}%")
    print(f"  Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"  Errors:   {metrics['total_errors']:,} / {metrics['total_diacritics']:,}")
    print(f"  Sequences: {metrics['num_sequences']:,}")

## 12. Training Functions

In [ ]:
print('cell 15')
# =============================================================================
# CELL 15: TRAINING FUNCTIONS WITH TIMING PROFILING
# =============================================================================

class TimingMeter:
    """Track timing statistics for profiling."""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.total = 0.0
        self.count = 0
    
    def update(self, time_val: float):
        self.total += time_val
        self.count += 1
    
    @property
    def avg(self) -> float:
        return self.total / self.count if self.count > 0 else 0.0


def train_epoch(model: nn.Module,
                train_loader: DataLoader,
                optimizer,
                scheduler,
                scaler: GradScaler,
                device: torch.device,
                epoch: int,
                use_fp16: bool = True,
                accumulation_steps: int = 1,
                log_interval: int = 50) -> Dict:
    """
    Train for one epoch with mixed precision, gradient accumulation, and timing profiling.
    
    Timing Profiling:
    - Data load time: Time spent waiting for the next batch from DataLoader
    - Train time: Time spent on forward pass, backward pass, and optimizer step
    
    Returns:
        Dict with training metrics including timing statistics
    """
    model.train()
    model.set_epoch(epoch)
    
    loss_meter = AverageMeter()
    data_time_meter = TimingMeter()
    train_time_meter = TimingMeter()
    
    epoch_start_time = time.time()
    
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]", leave=False)
    
    # Initialize data loading timer
    data_start = time.time()
    
    for batch_idx, batch in enumerate(pbar):
        # =====================================================================
        # TIMING: Data Loading Complete
        # =====================================================================
        data_time = time.time() - data_start
        data_time_meter.update(data_time)
        
        # =====================================================================
        # TIMING: Start Training Step
        # =====================================================================
        train_start = time.time()
        
        texts = batch['texts']
        chars_batch = batch['chars']
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        
        # Forward pass with mixed precision
        if use_fp16:
            with autocast('cuda'):
                loss = model(texts, chars_batch, labels, mask)
                loss = loss / accumulation_steps  # Scale loss for accumulation
            
            # Backward pass
            scaler.scale(loss).backward()
            
            # Step optimizer every accumulation_steps
            if (batch_idx + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
        else:
            loss = model(texts, chars_batch, labels, mask)
            loss = loss / accumulation_steps
            loss.backward()
            
            if (batch_idx + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
        
        # Synchronize GPU for accurate timing (important for CUDA)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        # =====================================================================
        # TIMING: Training Step Complete
        # =====================================================================
        train_time = time.time() - train_start
        train_time_meter.update(train_time)
        
        # Update metrics (unscaled loss)
        loss_meter.update(loss.item() * accumulation_steps)
        
        # =====================================================================
        # TIMING: Log every LOG_INTERVAL batches
        # =====================================================================
        if (batch_idx + 1) % log_interval == 0:
            total_time = data_time + train_time
            data_pct = (data_time / total_time * 100) if total_time > 0 else 0
            train_pct = (train_time / total_time * 100) if total_time > 0 else 0
            
            print(f"\n  Batch {batch_idx + 1}/{len(train_loader)}: "
                  f"Data {data_time:.3f}s ({data_pct:.1f}%) | "
                  f"Train {train_time:.3f}s ({train_pct:.1f}%) | "
                  f"Total {total_time:.3f}s | "
                  f"Loss {loss_meter.avg:.4f}")
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss_meter.avg:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}',
            'data': f'{data_time_meter.avg:.3f}s',
            'train': f'{train_time_meter.avg:.3f}s'
        })
        
        # =====================================================================
        # TIMING: Start next data loading measurement
        # =====================================================================
        data_start = time.time()
    
    # Handle remaining gradients if not divisible by accumulation_steps
    if (batch_idx + 1) % accumulation_steps != 0:
        if use_fp16:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
        scheduler.step()
    
    epoch_time = time.time() - epoch_start_time
    
    # =========================================================================
    # TIMING: Epoch Summary
    # =========================================================================
    print(f"\n  Epoch {epoch} Timing Summary:")
    print(f"    Avg Data Load Time: {data_time_meter.avg:.4f}s/batch")
    print(f"    Avg Train Time:     {train_time_meter.avg:.4f}s/batch")
    print(f"    Total Data Load:    {data_time_meter.total:.2f}s ({data_time_meter.total/epoch_time*100:.1f}%)")
    print(f"    Total Train Time:   {train_time_meter.total:.2f}s ({train_time_meter.total/epoch_time*100:.1f}%)")
    print(f"    Epoch Duration:     {epoch_time:.2f}s")
    
    return {
        'loss': loss_meter.avg,
        'time': epoch_time,
        'avg_data_time': data_time_meter.avg,
        'avg_train_time': train_time_meter.avg,
        'total_data_time': data_time_meter.total,
        'total_train_time': train_time_meter.total,
    }


@torch.no_grad()
def validate(model: nn.Module,
             val_loader: DataLoader,
             device: torch.device,
             use_fp16: bool = True) -> Dict:
    """
    Validate the model with timing.
    
    Returns:
        Dict with validation metrics including DER and timing
    """
    model.eval()
    
    loss_meter = AverageMeter()
    evaluator = Evaluator(ignore_no_diacritic=True)
    
    val_start = time.time()
    
    pbar = tqdm(val_loader, desc="Validating", leave=False)
    
    for batch in pbar:
        texts = batch['texts']
        chars_batch = batch['chars']
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        
        # Forward pass
        if use_fp16:
            with autocast('cuda'):
                loss = model(texts, chars_batch, labels, mask)
        else:
            loss = model(texts, chars_batch, labels, mask)
        
        loss_meter.update(loss.item())
        
        # Decode predictions
        predictions = model.decode(texts, chars_batch, mask)
        
        # Prepare targets for evaluation
        batch_size = labels.shape[0]
        targets_list = []
        masks_list = []
        
        for i in range(batch_size):
            seq_len = int(mask[i].sum().item())
            targets_list.append(labels[i, :seq_len].tolist())
            masks_list.append([True] * seq_len)
        
        evaluator.add_batch(predictions, targets_list, masks_list)
        
        pbar.set_postfix({'loss': f'{loss_meter.avg:.4f}'})
    
    val_time = time.time() - val_start
    
    metrics = evaluator.compute_metrics()
    metrics['loss'] = loss_meter.avg
    metrics['time'] = val_time
    
    return metrics

## 13. Main Training Loop

In [ ]:
print('cell 16')
# =============================================================================
# CELL 16: MAIN TRAINING LOOP WITH TIMING DIAGNOSTICS
# =============================================================================

def train_model(model: nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                config: Config,
                start_epoch: int = 1,
                optimizer_state: dict = None,
                scheduler_state: dict = None,
                scaler_state: dict = None,
                early_stopping_state: dict = None,
                best_der_resume: float = None) -> Dict:
    """
    Main training function with:
    - Differential learning rates
    - Linear warmup + linear decay scheduler
    - Mixed precision (FP16)
    - Early stopping
    - Model checkpointing (best_model.pt and last_model.pt)
    - Gradient accumulation
    - Resume training support
    - Timing profiling for bottleneck detection
    """
    device = DEVICE
    model = model.to(device)
    
    # Parameter groups with differential learning rates
    param_groups = model.get_parameter_groups(
        arabert_lr=config.ARABERT_LR,
        other_lr=config.LSTM_CRF_LR
    )
    
    # Optimizer
    optimizer = AdamW(param_groups, weight_decay=config.WEIGHT_DECAY)
    
    # Calculate total training steps
    accumulation_steps = config.GRADIENT_ACCUMULATION
    steps_per_epoch = len(train_loader) // accumulation_steps
    num_training_steps = steps_per_epoch * config.NUM_EPOCHS
    num_warmup_steps = int(num_training_steps * config.WARMUP_RATIO)
    
    # Scheduler: Linear warmup + linear decay
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Mixed precision scaler
    scaler = GradScaler('cuda', enabled=config.USE_FP16)
    
    # Load states if resuming training
    if optimizer_state is not None:
        optimizer.load_state_dict(optimizer_state)
        print("  ✓ Loaded optimizer state")
    if scheduler_state is not None:
        scheduler.load_state_dict(scheduler_state)
        print("  ✓ Loaded scheduler state")
    if scaler_state is not None and config.USE_FP16:
        scaler.load_state_dict(scaler_state)
        print("  ✓ Loaded scaler state")
    
    # Early stopping
    early_stopping = EarlyStopping(
        patience=config.EARLY_STOPPING_PATIENCE,
        mode='min'
    )
    
    # Restore early stopping state if resuming
    if early_stopping_state is not None:
        early_stopping.counter = early_stopping_state.get('counter', 0)
        early_stopping.best_score = early_stopping_state.get('best_score', None)
        early_stopping.best_epoch = early_stopping_state.get('best_epoch', 0)
        print(f"  ✓ Loaded early stopping state (best_epoch={early_stopping.best_epoch}, counter={early_stopping.counter})")
    
    # Training history (with timing)
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_der': [],
        'val_accuracy': [],
        'learning_rate': [],
        'avg_data_time': [],
        'avg_train_time': [],
    }
    
    # Initialize best_der
    best_der = best_der_resume if best_der_resume is not None else float('inf')
    best_model_state = None
    
    print("="*70)
    print("Starting Training (with Timing Profiling)")
    print("="*70)
    print(f"  Device: {device}")
    print(f"  Epochs: {config.NUM_EPOCHS}")
    print(f"  Batch size: {config.BATCH_SIZE}")
    print(f"  Max Sequence Length: {config.MAX_SEQ_LENGTH}")
    print(f"  Gradient Accumulation: {accumulation_steps}")
    print(f"  Effective Batch Size: {config.BATCH_SIZE * accumulation_steps}")
    print(f"  AraBERT LR: {config.ARABERT_LR}")
    print(f"  LSTM/CRF LR: {config.LSTM_CRF_LR}")
    print(f"  Warmup steps: {num_warmup_steps}")
    print(f"  Total steps: {num_training_steps}")
    print(f"  FP16: {config.USE_FP16}")
    print(f"  Bucketed Batching: Enabled")
    print(f"  Timing Log Interval: Every {config.LOG_INTERVAL} batches")
    if start_epoch > 1:
        print(f"  Resuming from epoch: {start_epoch}")
        print(f"  Best DER so far: {best_der*100:.2f}%")
    print("="*70)
    
    for epoch in range(start_epoch, config.NUM_EPOCHS + 1):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch}/{config.NUM_EPOCHS}")
        print(f"{'='*70}")
        
        # Train with timing
        train_metrics = train_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            device, epoch, config.USE_FP16, accumulation_steps,
            log_interval=config.LOG_INTERVAL
        )
        
        # Validate
        val_metrics = validate(model, val_loader, device, config.USE_FP16)
        
        # Get current learning rate
        current_lr = scheduler.get_last_lr()[0]
        
        # Update history
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_der'].append(val_metrics['der'])
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['learning_rate'].append(current_lr)
        history['avg_data_time'].append(train_metrics['avg_data_time'])
        history['avg_train_time'].append(train_metrics['avg_train_time'])
        
        # Print epoch summary
        print(f"\nEpoch {epoch} Summary:")
        print(f"  Train Loss: {train_metrics['loss']:.4f}")
        print(f"  Val Loss:   {val_metrics['loss']:.4f}")
        print(f"  Val DER:    {val_metrics['der']*100:.2f}%")
        print(f"  Val Acc:    {val_metrics['accuracy']*100:.2f}%")
        print(f"  LR:         {current_lr:.2e}")
        print(f"  Train Time: {train_metrics['time']:.1f}s (Val: {val_metrics['time']:.1f}s)")
        
        # Bottleneck diagnosis
        data_ratio = train_metrics['total_data_time'] / train_metrics['time'] * 100
        train_ratio = train_metrics['total_train_time'] / train_metrics['time'] * 100
        if data_ratio > 30:
            print(f"  ⚠ Data Loading Bottleneck Detected ({data_ratio:.1f}% of epoch time)")
            print(f"    → Consider increasing NUM_WORKERS (current: {config.NUM_WORKERS})")
        elif train_ratio > 90:
            print(f"  ✓ GPU-bound training ({train_ratio:.1f}% of epoch time) - Optimal!")
        
        # Check for best model
        is_best = early_stopping(val_metrics['der'], epoch)
        if is_best:
            best_der = val_metrics['der']
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ✓ New best model! DER: {best_der*100:.2f}%")
            
            # Save best_model.pt
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'early_stopping_state': {
                    'counter': early_stopping.counter,
                    'best_score': early_stopping.best_score,
                    'best_epoch': early_stopping.best_epoch,
                },
                'val_der': best_der,
                'best_der': best_der,
                'config': {
                    'arabert_model': config.ARABERT_MODEL_NAME,
                    'hidden_dim': config.LSTM_HIDDEN_SIZE,
                    'num_layers': config.LSTM_NUM_LAYERS,
                    'num_classes': NUM_DIACRITIC_CLASSES,
                    'dropout': config.LSTM_DROPOUT,
                }
            }, os.path.join(config.OUTPUT_DIR, 'best_model.pt'))
        
        # Save last_model.pt after every epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': {k: v.cpu().clone() for k, v in model.state_dict().items()},
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'early_stopping_state': {
                'counter': early_stopping.counter,
                'best_score': early_stopping.best_score,
                'best_epoch': early_stopping.best_epoch,
            },
            'val_der': val_metrics['der'],
            'best_der': best_der,
            'config': {
                'arabert_model': config.ARABERT_MODEL_NAME,
                'hidden_dim': config.LSTM_HIDDEN_SIZE,
                'num_layers': config.LSTM_NUM_LAYERS,
                'num_classes': NUM_DIACRITIC_CLASSES,
                'dropout': config.LSTM_DROPOUT,
            }
        }, os.path.join(config.OUTPUT_DIR, 'last_model.pt'))
        print(f"  ✓ Saved last_model.pt (epoch {epoch})")
        
        # Early stopping check
        if early_stopping.early_stop:
            print(f"\n⚠ Early stopping triggered at epoch {epoch}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # Final summary
    print("\n" + "="*70)
    print("Training Complete!")
    print("="*70)
    print(f"Best DER: {best_der*100:.2f}% (Epoch {early_stopping.best_epoch})")
    
    # Timing summary across all epochs
    print("\nTiming Summary (Averages Across All Epochs):")
    print(f"  Avg Data Load Time: {sum(history['avg_data_time'])/len(history['avg_data_time']):.4f}s/batch")
    print(f"  Avg Train Time:     {sum(history['avg_train_time'])/len(history['avg_train_time']):.4f}s/batch")
    
    return {
        'best_der': best_der,
        'best_epoch': early_stopping.best_epoch,
        'history': history
    }

---
## 📊 Understanding Timing Output & Bottleneck Detection

The training loop now prints detailed timing information to help you identify and fix performance bottlenecks.

### Timing Metrics Explained

| Metric | Description |
|--------|-------------|
| **Data Load Time** | Time spent waiting for the DataLoader to provide the next batch. Includes data reading, preprocessing, and transfer to GPU. |
| **Train Time** | Time spent on forward pass, backward pass, gradient clipping, and optimizer step. |

### How to Interpret the Output

Every `LOG_INTERVAL` batches (default: 50), you'll see:

```
Batch 50/156: Data 0.023s (8.5%) | Train 0.248s (91.5%) | Total 0.271s | Loss 2.3456
```

### Diagnosing Bottlenecks

#### ✅ **Optimal (GPU-bound)**
```
Data 0.02s (5-10%) | Train 0.25s (90-95%)
```
- **Meaning**: GPU is fully utilized; data loading is fast enough
- **Action**: No changes needed—you're at peak efficiency!

#### ⚠️ **Data Loading Bottleneck (CPU-bound)**
```
Data 0.15s (40%) | Train 0.22s (60%)
```
- **Meaning**: GPU is idle waiting for data
- **Actions to try**:
  1. Increase `NUM_WORKERS` (try 4 → 8)
  2. Increase `PREFETCH_FACTOR` (try 2 → 4)
  3. Ensure data is on fast storage (SSD, not HDD)
  4. On Kaggle, data should be in `/kaggle/input/`

#### ⚠️ **GPU Memory/Compute Bottleneck**
```
Data 0.01s (2%) | Train 0.80s (98%)
```
- **Meaning**: Training is slow, possibly due to large batches or sequences
- **Actions to try**:
  1. Reduce `BATCH_SIZE` (64 → 32)
  2. Reduce `MAX_SEQ_LENGTH` (256 → 128)
  3. Enable gradient accumulation if batch is too small

### Quick Reference Table

| Symptom | Likely Cause | Solution |
|---------|--------------|----------|
| Data % > 30% | Slow data loading | ↑ `NUM_WORKERS`, ↑ `PREFETCH_FACTOR` |
| Train time very high | Large batch/sequence | ↓ `BATCH_SIZE`, ↓ `MAX_SEQ_LENGTH` |
| GPU utilization < 80% | Batch too small | ↑ `BATCH_SIZE` |
| OOM errors | Batch/sequence too large | ↓ `BATCH_SIZE`, ↓ `MAX_SEQ_LENGTH` |

### Bucketed Batching Benefits

With bucketed batching enabled, you should see:
- **More consistent batch times** (less variance in Train time)
- **Reduced padding overhead** (similar-length sequences grouped together)
- **Better GPU memory utilization** (no wasted computation on padding)

### Monitoring GPU Utilization (Kaggle)

To check GPU utilization in a Kaggle notebook, add a cell:
```python
!nvidia-smi
```

Look for:
- **GPU-Util**: Should be 80-100% during training
- **Memory-Usage**: Should be high but not 100%

## 14. Execute Training

In [ ]:
print('cell 17')
# =============================================================================
# CELL 17: INITIALIZE AND RUN TRAINING
# =============================================================================

# =============================================================================
# RESUME TRAINING INSTRUCTIONS
# =============================================================================
# To resume training from a checkpoint, set RESUME_TRAINING = True and specify
# the checkpoint path. The checkpoint contains all states needed:
# - model_state_dict: Model weights
# - optimizer_state_dict: Optimizer state (momentum, etc.)
# - scheduler_state_dict: Learning rate scheduler state
# - scaler_state_dict: Mixed precision scaler state  
# - early_stopping_state: Early stopping counter and best score
# - best_der: Best validation DER achieved so far
# - epoch: Last completed epoch
#
# Example:
#   RESUME_TRAINING = True
#   CHECKPOINT_PATH = 'last_model.pt'  # or 'best_model.pt'
# =============================================================================

RESUME_TRAINING = False  # Set to True to resume from checkpoint
CHECKPOINT_PATH = 'last_model.pt'  # Path to checkpoint file

if __name__ == "__main__" or True:  # Always run in notebook
    
    # Configuration
    config = Config()
    
    # Print configuration
    print("Configuration:")
    print(f"  ARABERT_MODEL: {config.ARABERT_MODEL_NAME}")
    print(f"  LSTM_HIDDEN_SIZE: {config.LSTM_HIDDEN_SIZE}")
    print(f"  LSTM_NUM_LAYERS: {config.LSTM_NUM_LAYERS}")
    print(f"  LSTM_DROPOUT: {config.LSTM_DROPOUT}")
    print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
    print(f"  GRADIENT_ACCUMULATION: {config.GRADIENT_ACCUMULATION}")
    print(f"  EFFECTIVE_BATCH_SIZE: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")
    print(f"  NUM_EPOCHS: {config.NUM_EPOCHS}")
    print(f"  ARABERT_LR: {config.ARABERT_LR}")
    print(f"  LSTM_CRF_LR: {config.LSTM_CRF_LR}")
    print(f"  USE_FP16: {config.USE_FP16}")
    print(f"  EARLY_STOPPING_PATIENCE: {config.EARLY_STOPPING_PATIENCE}")
    print(f"  NUM_WORKERS: {config.NUM_WORKERS}")
    print(f"  MAX_SEQ_LENGTH: {config.MAX_SEQ_LENGTH}")
    print()
    
    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    
    # Load data
    print("Loading dataset...")
    train_data = process_dataset(config.TRAIN_FILE, verbose=True)
    val_data = process_dataset(config.VAL_FILE, verbose=True)
    
    # Create datasets
    print("\nCreating datasets...")
    train_dataset = DiacritizationDataset(train_data)
    val_dataset = DiacritizationDataset(val_data)
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Val: {len(val_dataset)} samples")
    
    # Create dataloaders
    print("\nCreating dataloaders...")
    train_loader, val_loader = create_dataloaders(
        train_dataset, val_dataset,
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS
    )
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    
    # Initialize model
    print("\nInitializing model...")
    model = ArabicDiacritizationModel(
        arabert_model=config.ARABERT_MODEL_NAME,
        hidden_dim=config.LSTM_HIDDEN_SIZE,
        num_layers=config.LSTM_NUM_LAYERS,
        num_classes=NUM_DIACRITIC_CLASSES,
        dropout=config.LSTM_DROPOUT,
        use_char_features=True
    )
    
    # Single GPU training
    print(f"  Using device: {DEVICE}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
    # ==========================================================================
    # RESUME TRAINING LOGIC
    # ==========================================================================
    start_epoch = 1
    optimizer_state = None
    scheduler_state = None
    scaler_state = None
    early_stopping_state = None
    best_der_resume = None
    
    if RESUME_TRAINING:
        checkpoint_full_path = os.path.join(config.OUTPUT_DIR, CHECKPOINT_PATH)
        if os.path.exists(checkpoint_full_path):
            print(f"\n{'='*60}")
            print(f"RESUMING TRAINING FROM: {checkpoint_full_path}")
            print(f"{'='*60}")
            
            checkpoint = torch.load(checkpoint_full_path, map_location=DEVICE)
            
            # Load model weights
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"  ✓ Loaded model weights")
            
            # Get resume states
            start_epoch = checkpoint.get('epoch', 0) + 1
            optimizer_state = checkpoint.get('optimizer_state_dict', None)
            scheduler_state = checkpoint.get('scheduler_state_dict', None)
            scaler_state = checkpoint.get('scaler_state_dict', None)
            early_stopping_state = checkpoint.get('early_stopping_state', None)
            best_der_resume = checkpoint.get('best_der', None)
            
            print(f"  ✓ Will resume from epoch {start_epoch}")
            if best_der_resume is not None:
                print(f"  ✓ Best DER so far: {best_der_resume*100:.2f}%")
            print(f"{'='*60}\n")
        else:
            print(f"\n⚠ Checkpoint not found: {checkpoint_full_path}")
            print("  Starting fresh training instead.\n")
    
    # Train
    print("\n" + "="*60)
    results = train_model(
        model, train_loader, val_loader, config,
        start_epoch=start_epoch,
        optimizer_state=optimizer_state,
        scheduler_state=scheduler_state,
        scaler_state=scaler_state,
        early_stopping_state=early_stopping_state,
        best_der_resume=best_der_resume
    )
    
    # Save final model
    final_model_path = os.path.join(config.OUTPUT_DIR, 'final_model.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': {
            'arabert_model': config.ARABERT_MODEL_NAME,
            'hidden_dim': config.LSTM_HIDDEN_SIZE,
            'num_layers': config.LSTM_NUM_LAYERS,
            'num_classes': NUM_DIACRITIC_CLASSES,
            'dropout': config.LSTM_DROPOUT,
            'aggregation_method': config.ARABERT_AGGREGATION
        },
        'best_der': results['best_der'],
        'best_epoch': results['best_epoch'],
        'diacritic_to_id': DIACRITIC_TO_ID,
        'id_to_diacritic': ID_TO_DIACRITIC
    }, final_model_path)
    print(f"\nFinal model saved to: {final_model_path}")
    
    # Save training history
    history_path = os.path.join(config.OUTPUT_DIR, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(results['history'], f, indent=2)
    print(f"Training history saved to: {history_path}")

## 15. Training Visualization & Summary

In [ ]:
print('cell 18')
# =============================================================================
# CELL 18: PLOT TRAINING CURVES WITH TIMING
# =============================================================================

import matplotlib.pyplot as plt

def plot_training_history(history: Dict):
    """Plot training curves including timing metrics."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # DER
    ax = axes[0, 1]
    der_percent = [d * 100 for d in history['val_der']]
    ax.plot(epochs, der_percent, 'g-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('DER (%)')
    ax.set_title('Validation Diacritic Error Rate')
    ax.grid(True, alpha=0.3)
    best_epoch = der_percent.index(min(der_percent)) + 1
    ax.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.7, label=f'Best: {min(der_percent):.2f}%')
    ax.legend()
    
    # Accuracy
    ax = axes[0, 2]
    acc_percent = [a * 100 for a in history['val_accuracy']]
    ax.plot(epochs, acc_percent, 'm-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Validation Accuracy')
    ax.grid(True, alpha=0.3)
    
    # Learning Rate
    ax = axes[1, 0]
    ax.plot(epochs, history['learning_rate'], 'c-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    # Timing: Data Load vs Train Time
    ax = axes[1, 1]
    if 'avg_data_time' in history and 'avg_train_time' in history:
        ax.bar(epochs, history['avg_data_time'], label='Data Load', alpha=0.7, color='orange')
        ax.bar(epochs, history['avg_train_time'], bottom=history['avg_data_time'], 
               label='Train', alpha=0.7, color='blue')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Time (s/batch)')
        ax.set_title('Batch Timing Breakdown')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Timing data not available', ha='center', va='center', transform=ax.transAxes)
    
    # Timing Ratio
    ax = axes[1, 2]
    if 'avg_data_time' in history and 'avg_train_time' in history:
        total_times = [d + t for d, t in zip(history['avg_data_time'], history['avg_train_time'])]
        data_pct = [d / t * 100 if t > 0 else 0 for d, t in zip(history['avg_data_time'], total_times)]
        train_pct = [t / tot * 100 if tot > 0 else 0 for t, tot in zip(history['avg_train_time'], total_times)]
        
        ax.fill_between(epochs, 0, data_pct, alpha=0.7, label='Data Load %', color='orange')
        ax.fill_between(epochs, data_pct, [d + t for d, t in zip(data_pct, train_pct)], 
                        alpha=0.7, label='Train %', color='blue')
        ax.axhline(y=30, color='r', linestyle='--', alpha=0.5, label='Bottleneck Threshold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Percentage (%)')
        ax.set_title('Time Distribution (Data vs Train)')
        ax.set_ylim(0, 100)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Timing data not available', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'training_curves.png'), dpi=150)
    plt.show()
    print(f"\nTraining curves saved to: {os.path.join(config.OUTPUT_DIR, 'training_curves.png')}")

# Plot if training completed
if 'results' in dir() and 'history' in results:
    plot_training_history(results['history'])

---

## 🎉 Training Complete!

### Output Files
After training completes, you will have the following files in the output directory:

| File | Description |
|------|-------------|
| `best_model.pt` | Best model checkpoint (lowest DER) with full state for resume |
| `last_model.pt` | Latest checkpoint after each epoch for resume training |
| `final_model.pt` | Final model with full configuration |
| `training_history.json` | Training metrics per epoch |
| `training_curves.png` | Visualization of training progress |

### Model Architecture
- **AraBERT**: `aubmindlab/bert-base-arabertv02` (768-dim embeddings)
- **BiLSTM**: 2 layers, 512 hidden units, bidirectional (1024-dim output)
- **CRF**: Conditional Random Field for sequence labeling
- **Classes**: 16 diacritic classes

### Hyperparameters Used
| Parameter | Value |
|-----------|-------|
| Batch Size | 128 |
| Gradient Accumulation | 1 |
| Effective Batch Size | 128 |
| Epochs | 8 |
| LSTM Layers | 2 |
| Max Sequence Length | 512 |
| AraBERT LR | 2e-5 |
| LSTM/CRF LR | 1e-3 |
| Early Stopping Patience | 7 |
| Mixed Precision (FP16) | Enabled |
| Num Workers | 4 |

### Checkpoint Contents
Both `best_model.pt` and `last_model.pt` contain:
- `model_state_dict`: Model weights
- `optimizer_state_dict`: Optimizer state
- `scheduler_state_dict`: Learning rate scheduler state
- `scaler_state_dict`: Mixed precision scaler state
- `early_stopping_state`: Early stopping counter and best score
- `best_der`: Best validation DER achieved
- `epoch`: Epoch number
- `config`: Model configuration

### Resume Training
To resume training from a checkpoint:

```python
# In Cell 17, set:
RESUME_TRAINING = True
CHECKPOINT_PATH = 'last_model.pt'  # or 'best_model.pt'

# Then run Cell 17 - it will automatically:
# 1. Load model weights
# 2. Restore optimizer state (momentum, etc.)
# 3. Restore scheduler state (learning rate position)
# 4. Restore scaler state (mixed precision)
# 5. Restore early stopping state (counter, best score)
# 6. Continue from the next epoch
```

### Loading the Trained Model for Inference
```python
checkpoint = torch.load('best_model.pt', map_location=DEVICE)
model = ArabicDiacritizationModel(
    arabert_model=checkpoint['config']['arabert_model'],
    hidden_dim=checkpoint['config']['hidden_dim'],
    num_layers=checkpoint['config']['num_layers'],
    num_classes=checkpoint['config']['num_classes'],
    dropout=checkpoint['config']['dropout']
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded model with DER: {checkpoint['best_der']*100:.2f}%")
```